In [ ]:
from cube_nn import CubeValueResNet
from training import train_on_value_dataset
from torch import optim
import matplotlib.pyplot as plt
import torch
from cube_nn import NNValueFunctionType

device = "cuda"

In [ ]:
# Create a new net
net = CubeValueResNet()
net = net.to(device)

losses = []
i = 0

In [ ]:
# Load a previously saved net
i = 0
net = CubeValueResNet()
net.load_state_dict(torch.load(f'temp_models/resnet2.1/cube_value_resnet_iter_{i}.pth'))


losses = torch.load(f'temp_models/resnet2.1/losses_list_{i}.pth')

In [ ]:
from concurrent.futures import ProcessPoolExecutor

N_CUBES_PER_MOVE = 1000
N_ROLLOUTS = 20000
N_MOVES_MAX = 20
N_MOVES_DIFFERENCE_MAX = 3
N_EPOCHS = 1
BATCH_SIZE = 100
SIMILAR_EXTENSION_FRACTION = None

# set the STANDARD value function for speed
net.set_value_function_type(NNValueFunctionType.STANDARD)
net.to(device)
optimizer = optim.Adam(net.parameters(), lr=0.001)

def generate_dataset_worker(iteration, n_rollouts, n_moves_max):
    """Worker function to generate dataset on CPU while GPU trains
    
    This runs in a separate process to avoid GIL contention with the main training loop.
    Uses tqdm position 1 for progress bar to avoid conflicts with training (position 0).
    """
    from cube_datasets import TrainingValueDataset
    from cube_nn import cube_to_tensor_one_hot
    
    print(f"[Dataset Gen] Starting generation for iteration {iteration}")
    new_dataset = TrainingValueDataset.create_from_trajectories(
        cube_to_tensor_one_hot,
        n_rollouts, 
        n_moves_max, 
        device="cpu",
        tqdm_position=1  # Use position 1 for dataset generation
    )
    print(f"[Dataset Gen] Completed generation for iteration {iteration}")
    
    return new_dataset

# Use ProcessPoolExecutor for parallel execution
with ProcessPoolExecutor(max_workers=1) as executor:
    # Start generating the FIRST dataset in background
    print("Starting generation of first dataset...")
    future_dataset = executor.submit(generate_dataset_worker, i+1, N_ROLLOUTS, N_MOVES_MAX)
    
    for j in range(20):
        i += 1
        
        # Wait for current dataset to be ready and move it to GPU
        print(f"[Training] Waiting for dataset {i}...")
        current_dataset = future_dataset.result()
        current_dataset._inputs = current_dataset._inputs.to(device)
        current_dataset._targets = current_dataset._targets.to(device)
        print(f"[Training] Dataset {i} ready, moving to GPU")
        
        # Immediately start generating the NEXT dataset in parallel (before training starts)
        if j < 19:  # Don't generate after the last iteration
            future_dataset = executor.submit(generate_dataset_worker, i+2, N_ROLLOUTS, N_MOVES_MAX)
            print(f"[Dataset Gen] Started background generation for iteration {i+2}")
        
        # Train on current dataset (on GPU) - use position 0 for training progress bars
        print(f"[Training] Starting training iteration {i}")
        loss = train_on_value_dataset(net, current_dataset, optimizer, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, device=device, tqdm_position=0)
        losses.append(loss)
        print(f"Iteration {i}, loss: {loss}")
        
        # Save the model and plot of losses every 10 iterations
        if (i + 1) % 10 == 0:
            torch.save(net.state_dict(), f'temp_models/resnet2.1/cube_value_resnet_iter_{i+1}.pth')
            torch.save(losses, f'temp_models/resnet2.1/losses_list_{i+1}.pth')
            plt.plot(losses[1:]) # Skip initial loss for better visualization
            plt.xlabel('Iteration')
            plt.ylabel('Loss')
            plt.title('Training Loss Over Iterations')
            plt.savefig(f'figs/resnet2.1/loss_plot_iter_{i+1}.png')
            plt.clf()

print("Training completed!")
